In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))


In [44]:
import json
from pathlib import Path
from typing import Dict, Optional, Union

import anndata as ad
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

from papers.scgpt.scgpt.tokenizer import GeneVocab
from papers.scgpt.scgpt.model import TransformerModel
from papers.scgpt.scgpt.utils import load_pretrained


In [4]:
def get_scgpt_model(
    model_dir: str,
    device: str = 'cuda',
    use_fast_transformer: bool = True,
) -> torch.nn.Module:
    """
    Preprocess anndata and embed the data using the model.

    Args:
        adata_or_file (Union[AnnData, PathLike]): The AnnData object or the path to the
            AnnData object.
        model_dir (PathLike): The path to the model directory.
        gene_col (str): The column in adata.var that contains the gene names.
        max_length (int): The maximum length of the input sequence. Defaults to 1200.
        batch_size (int): The batch size for inference. Defaults to 64.
        obs_to_save (Optional[list]): The list of obs columns to save in the output adata.
            Useful for retaining meta data to output. Defaults to None.
        device (Union[str, torch.device]): The device to use. Defaults to "cuda".
        use_fast_transformer (bool): Whether to use flash-attn. Defaults to True.
        return_new_adata (bool): Whether to return a new AnnData object. If False, will
            add the cell embeddings to a new :attr:`adata.obsm` with key "X_scGPT".

    Returns:
        AnnData: The AnnData object with the cell embeddings.
    """
    # LOAD MODEL
    model_dir = Path(model_dir)
    vocab_file = model_dir / "vocab.json"
    model_config_file = model_dir / "args.json"
    model_file = model_dir / "best_model.pt"
    pad_token = "<pad>"
    special_tokens = [pad_token, "<cls>", "<eoc>"]

    # vocabulary
    vocab = GeneVocab.from_file(vocab_file)
    for s in special_tokens:
        if s not in vocab:
            vocab.append_token(s)

    with open(model_config_file, "r") as f:
        model_configs = json.load(f)

    vocab.set_default_index(vocab["<pad>"])
    model = TransformerModel(
        ntoken=len(vocab),
        d_model=model_configs["embsize"],
        nhead=model_configs["nheads"],
        d_hid=model_configs["d_hid"],
        nlayers=model_configs["nlayers"],
        nlayers_cls=model_configs["n_layers_cls"],
        n_cls=1,
        vocab=vocab,
        dropout=model_configs["dropout"],
        pad_token=model_configs["pad_token"],
        pad_value=model_configs["pad_value"],
        do_mvc=True,
        do_dab=False,
        use_batch_labels=False,
        domain_spec_batchnorm=False,
        explicit_zero_prob=False,
        use_fast_transformer=use_fast_transformer,
        fast_transformer_backend="flash",
        pre_norm=False,
    )
    load_pretrained(model, torch.load(model_file, map_location=device), verbose=False)
    model.to(device)
    model.eval()

    return model, vocab


In [32]:
model_path = '../papers/scgpt/save/whole_human'
data_path = 'data_new/test.h5ad'

model, vocab = get_scgpt_model(model_path, device='cuda')
adata = ad.read_h5ad(data_path)


/storage/scratch/2370352/my-research/papers/scgpt/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(


In [27]:
def _digitize(x: np.ndarray, bins: np.ndarray, side="both") -> np.ndarray:
    """
    Digitize the data into bins. This method spreads data uniformly when bins
    have same values.

    Args:

    x (:class:`np.ndarray`):
        The data to digitize.
    bins (:class:`np.ndarray`):
        The bins to use for digitization, in increasing order.
    side (:class:`str`, optional):
        The side to use for digitization. If "one", the left side is used. If
        "both", the left and right side are used. Default to "one".

    Returns:

    :class:`np.ndarray`:
        The digitized data.
    """
    assert x.ndim == 1 and bins.ndim == 1

    left_digits = np.digitize(x, bins)
    if side == "one":
        return left_digits

    right_difits = np.digitize(x, bins, right=True)

    rands = np.random.rand(len(x))  # uniform random numbers

    digits = rands * (right_difits - left_digits) + left_digits
    digits = np.ceil(digits).astype(np.int64)
    return digits


def binning(
    row: Union[np.ndarray, torch.Tensor], n_bins: int
) -> Union[np.ndarray, torch.Tensor]:
    """Binning the row into n_bins."""
    dtype = row.dtype
    return_np = False if isinstance(row, torch.Tensor) else True
    row = row.cpu().numpy() if isinstance(row, torch.Tensor) else row

    if row.max() == 0:
        return (
            np.zeros_like(row, dtype=dtype)
            if return_np
            else torch.zeros_like(row, dtype=dtype)
        )

    if row.min() <= 0:
        non_zero_ids = row.nonzero()
        non_zero_row = row[non_zero_ids]
        bins = np.quantile(non_zero_row, np.linspace(0, 1, n_bins - 1))
        non_zero_digits = _digitize(non_zero_row, bins)
        binned_row = np.zeros_like(row, dtype=np.int64)
        binned_row[non_zero_ids] = non_zero_digits
    else:
        bins = np.quantile(row, np.linspace(0, 1, n_bins - 1))
        binned_row = _digitize(row, bins)
    return torch.from_numpy(binned_row) if not return_np else binned_row.astype(dtype)


In [20]:
class AnnDataDataset(Dataset):
    def __init__(self, adata, vocab, max_genes=2000, n_bins=51):
        self.adata = adata
        self.X = adata.X.tocsr()
        self.max_genes = max_genes
        self.n_bins = n_bins

        # GeneVocab → dict
        self.gene2id = vocab.stoi if hasattr(vocab, "stoi") else vocab.get_stoi()

        # map var_names → vocab ids
        self.var_to_vocab = np.array([
            self.gene2id.get(g, self.gene2id.get("<unk>", 0))
            for g in adata.var['feature_name'].tolist()
        ])
        self.cell_types = adata.obs["cell_type"].astype("category")
        self.label2id = {k: i for i, k in enumerate(self.cell_types.cat.categories)}
        self.labels = self.cell_types.cat.codes.values.astype(np.int64)
        
    def __len__(self):
        return self.adata.n_obs

    def __getitem__(self, idx):
        row = self.X[idx]

        gene_ids = self.var_to_vocab[row.indices]
        values = row.data.astype(np.float32)

        # top-k genes
        if len(values) > self.max_genes:
            topk = np.argsort(values)[-self.max_genes:]
            gene_ids = gene_ids[topk]
            values = values[topk]

        src = torch.tensor(gene_ids, dtype=torch.long)
        values = torch.tensor(values, dtype=torch.float32)
        values = binning(values, self.n_bins)
        mask = torch.zeros(len(src), dtype=torch.bool)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return src, values, mask, label


def collate_fn(batch):
    srcs, values, _, label = zip(*batch)
    max_len = max(len(x) for x in srcs)
    batch_size = len(batch)

    src_pad = torch.zeros(batch_size, max_len, dtype=torch.long)
    val_pad = torch.zeros(batch_size, max_len, dtype=torch.float)
    mask_pad = torch.ones(batch_size, max_len, dtype=torch.bool)  # True = padding

    for i in range(batch_size):
        L = len(srcs[i])

        src_pad[i, :L] = srcs[i]
        val_pad[i, :L] = values[i]
        mask_pad[i, :L] = False 

    return src_pad, val_pad, mask_pad, label


In [31]:
dataset = AnnDataDataset(adata, vocab, max_genes=2000)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0
)

batch = next(iter(loader))
src, values, mask, labels = batch

out = model(src, values, mask)
emb = out["cell_emb"]
emb

tensor([[ 0.1582,  0.5548, -0.6197,  ..., -0.0896, -0.3550,  0.2001],
        [ 0.1713,  0.8922, -0.9919,  ..., -0.1544, -0.4884, -0.7698],
        [ 0.2617,  0.7843, -0.5383,  ...,  0.1982,  0.3722, -0.3893],
        ...,
        [-0.0581,  0.1454, -0.5944,  ...,  0.4241,  0.0998,  0.2231],
        [ 0.1633,  0.1580, -0.3949,  ...,  0.1387, -0.1971, -0.0532],
        [ 0.6597,  0.3885, -0.9622,  ...,  0.1653,  0.6472, -0.1480]],
       grad_fn=<SliceBackward0>)

In [49]:
all_emb = []
all_labels = []

model.eval()

with torch.no_grad():
    for src, values, mask, labels in tqdm(loader):
        out = model(src.cuda(), values.cuda(), mask.cuda())
        emb = out["cell_emb"]

        all_emb.append(emb.cpu())

        # 🔥 FIX: labels tuple → concat
        if isinstance(labels, (tuple, list)):
            labels = torch.stack(labels)

        all_labels.append(labels.cpu())

all_emb = torch.cat(all_emb, dim=0).numpy()
all_labels = torch.cat(all_labels, dim=0).numpy()


100%|██████████| 44/44 [00:59<00:00,  1.34s/it]
